# Waste detector evaluation in kaggle
if you wanna try yourself to evaluate our model or model that u made from our dataset - just import this file and run the codes one by one. 
the model should be imported (5th) + our 20k images dataset

In [ ]:
%pip install ultralytics==8.4.208


In [ ]:
from pathlib import Path
import shutil, yaml
from ultralytics import YOLO

DATA = Path("/kaggle/input/datasets/hlibstankevych/computah-vision")
# Attach the selected checkpoint as a Kaggle input. It may be called best.pt or best.zip.
found = [p for p in Path("/kaggle/input").rglob("*") if p.is_file() and p.name.lower() in ("best.pt", "best.zip", "best (1).zip")]
print("Model candidates:", *found, sep="\n")
if len(found) != 1:
    raise RuntimeError("Attach exactly one selected model; if needed set the path manually.")
for split in ("train", "valid", "test"):
    for sub in ("images", "labels"):
        p = DATA / split / sub
        if not p.is_dir(): raise FileNotFoundError(p)
    print(split, len(list((DATA / split / "images").glob("*.jpg"))), "jpg images")
config = {"path": str(DATA), "train": "train/images", "val": "valid/images", "test": "test/images", "names": {0:"biodegradable",1:"cardboard",2:"glass",3:"metal",4:"paper",5:"plastic"}}
config_path = Path("/kaggle/working/waste_eval.yaml")
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
# The downloaded .zip is a PyTorch checkpoint container; copy without altering the original.
model_path = Path("/kaggle/working/selected_best.pt")
shutil.copy2(found[0], model_path)
print("Checkpoint:", found[0], "bytes:", found[0].stat().st_size)
model = YOLO(str(model_path))
print("Model classes:", model.names)
assert [model.names[i] for i in range(6)] == list(config["names"].values()), "Class mapping mismatch"


## Validation: run first
Prints class metrics from the existing validation split. A re-evaluation may differ slightly from logged training values.


In [ ]:
val = model.val(data=str(config_path), split="val", imgsz=512, device=0, max_det=358, plots=True, project="/kaggle/working/waste_evaluation", name="validation")
print("Overall:", val.results_dict)
for i, name in model.names.items():
    print(name, "AP50:", round(float(val.box.ap50[i]), 4), "AP50-95:", round(float(val.box.maps[i]), 4))


## Final test
Only run this after the model and settings have been chosen using validation. Do not tune the model using test results


In [ ]:
test = model.val(data=str(config_path), split="test", imgsz=512, device=0, max_det=358, plots=True, project="/kaggle/working/waste_evaluation", name="test")
print("Overall:", test.results_dict)
for i, name in model.names.items():
    print(name, "AP50:", round(float(test.box.ap50[i]), 4), "AP50-95:", round(float(test.box.maps[i]), 4))


## Example detections
Search the test set for examples. Correct / missed / incorrect are defined at confidence 0.25 and IoU 0.50 with class-aware one-to-one greedy matching. Manually review the images and labels before presenting them; a missed annotation or ambiguous class can affect these categories.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

CONF, MATCH_IOU = 0.25, 0.50
def iou(a,b):
    x1,y1=max(a[0],b[0]),max(a[1],b[1]); x2,y2=min(a[2],b[2]),min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    return inter / max((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter, 1e-9)

output = Path("/kaggle/working/waste_evaluation/examples")
output.mkdir(parents=True, exist_ok=True)
selected = {"correct": [], "missed": [], "incorrect": []}
paths = sorted(p for p in (DATA / "test/images").iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
for image_path in paths:
    label_path = DATA / "test/labels" / (image_path.stem + ".txt")
    if not label_path.exists(): continue
    with Image.open(image_path) as im: w,h=im.size
    gt=[]
    for line in label_path.read_text().splitlines():
        v=line.split()
        if len(v)!=5: continue
        cls=int(v[0]); x,y,bw,bh=map(float,v[1:]); gt.append((cls, [(x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h]))
    pred=model.predict(str(image_path), imgsz=512, conf=CONF, max_det=358, device=0, verbose=False)[0]
    boxes=pred.boxes.xyxy.cpu().numpy(); classes=pred.boxes.cls.cpu().numpy().astype(int); scores=pred.boxes.conf.cpu().numpy()
    used=set(); matches=0; false_pos=0
    for j in np.argsort(-scores):
        choices=[(iou(boxes[j], box), k) for k,(cls,box) in enumerate(gt) if cls==classes[j] and k not in used]
        overlap,k=max(choices, default=(0,-1))
        if overlap>=MATCH_IOU: used.add(k); matches+=1
        else: false_pos+=1
    missed=len(gt)-len(used)
    cats=[]
    if gt and matches and not missed and not false_pos: cats.append("correct")
    if missed: cats.append("missed")
    if false_pos: cats.append("incorrect")
    cats=[c for c in cats if len(selected[c])<3]
    if not cats: continue
    fig,ax=plt.subplots(figsize=(9,9)); ax.imshow(Image.open(image_path).convert("RGB")); ax.axis("off")
    for cls,(x1,y1,x2,y2) in gt:
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor="lime",linewidth=1.2))
        ax.text(x1,y1,str(model.names[cls]),color="lime",fontsize=6,backgroundcolor="black")
    for cls,box,score in zip(classes,boxes,scores):
        x1,y1,x2,y2=box; ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor="red",linewidth=1.2))
        ax.text(x1,y2,f"{model.names[cls]} {score:.2f}",color="red",fontsize=6,backgroundcolor="black")
    ax.set_title(f"{image_path.name} | ground truth: green, prediction: red")
    for category in cats:
        dest=output / f"{category}_{len(selected[category])+1}.png"; fig.savefig(dest,bbox_inches="tight",dpi=130); selected[category].append(image_path.name)
    plt.close(fig)
    if all(len(v)>=3 for v in selected.values()): break
print("Selected examples:", selected)
print("Saved to", output)


After completion, download `/kaggle/working/waste_evaluation` including metrics, plots, and examples. Record the source image names.
